In [1]:
import pandas as pd
from Python import config, features

pitcher_training = pd.read_parquet(config.PITCHER_TRAINING_PATH)
pitcher_training.shape

(14124, 242)

In [2]:
print(sorted(features.LABEL_COLUMNS))
print(sorted(features.MODEL_METADATA_COLUMNS))
print(sorted(features.FORBIDDEN_PREGAME_FEATURES))

['K', 'Outs', 'PA', 'k_rate']
['away_team', 'bat_team', 'batter', 'batter_name', 'game_date', 'game_pk', 'home_team', 'is_initial_lineup', 'opp_lineup_size', 'opp_team', 'p_throws', 'pitcher', 'pitcher_name', 'player_name', 'season', 'stand']
['K', 'Outs', 'PA', 'actual_k', 'actual_outs', 'actual_pa', 'actual_tbf', 'k_rate']


In [3]:
selected = features.model_feature_names(pitcher_training)
len(selected)
selected[:20]

('is_home',
 'k_rate_P5',
 'k_rate_P10',
 'k_rate_P20',
 'k_rate_std',
 'bb_rate_P5',
 'bb_rate_P10',
 'bb_rate_P20',
 'bb_rate_std',
 'csw_rate_P5',
 'csw_rate_P10',
 'csw_rate_P20',
 'csw_rate_std',
 'swstr_rate_P5',
 'swstr_rate_P10',
 'swstr_rate_P20',
 'swstr_rate_std',
 'whiff_rate_P5',
 'whiff_rate_P10',
 'whiff_rate_P20')

In [4]:
FROZEN_FEATURE_COUNT = 227
assert len(selected) == FROZEN_FEATURE_COUNT, (
    f"feature count drifted: expected {FROZEN_FEATURE_COUNT}, got {len(selected)}"
)

In [5]:
selected_set = set(selected)
leaked_labels = selected_set & features.LABEL_COLUMNS
leaked_metadata = selected_set & features.MODEL_METADATA_COLUMNS
leaked_labels, leaked_metadata

(set(), set())

In [6]:
expected_new_cols = {
    "opp_lineup_k", "opp_lineup_k_vs_hand",
    "opp_lineup_whiff", "opp_lineup_chase", "park_k_factor",
}
expected_new_cols - selected_set

set()

In [7]:
dtypes = pitcher_training[list(selected)].dtypes
dtypes[~dtypes.apply(lambda d: pd.api.types.is_numeric_dtype(d) or pd.api.types.is_bool_dtype(d))]

Series([], dtype: object)

In [8]:
try:
    features.validate_pregame_features(["k_rate_P5", "k_rate_P5", "bb_rate_std"])
except ValueError as e:
    print(e)

Invalid pregame feature list (duplicate features: ['k_rate_P5'])


In [9]:
try:
    features.validate_pregame_features(["k_rate_P5", "K", "PA"])
except ValueError as e:
    print(e)

Invalid pregame feature list (same-game features: ['K', 'PA'])


In [10]:
set(pitcher_training.columns) & features.FORBIDDEN_PREGAME_FEATURES

{'K', 'Outs', 'PA', 'k_rate'}

In [11]:
batter_training = pd.read_parquet(config.BATTER_TRAINING_PATH)
batter_selected = features.model_feature_names(batter_training)
len(batter_selected)
set(batter_selected) & features.LABEL_COLUMNS

set()